In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "01_data_pipeline_v1"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-25"
UPDATED = "2026-03-25"
PROJECT = "macro-market-analysis"
STAGE = "Data Processing"
VERSION = "0.1.0"

In [ ]:
# ============================================================
# PURPOSE
# ============================================================

PURPOSE = """
Transform raw market data into processed datasets.

This notebook triggers the processed data pipeline, which:

- normalizes raw data
- applies corporate action adjustments
- constructs total return price series
- computes log returns (multi-horizon)
- computes liquidity features
- enforces schema consistency
- saves datasets in data/processed/

This is the canonical dataset used by all research layers.
"""

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
from quant_research.data.processed.pipelines.data_processing_pipeline import run_data_processing_pipeline
from quant_research.config.paths import PROCESSED_DATA_PATH

In [ ]:
# ============================================================
# RUN DATA PROCESSING PIPELINE
# ============================================================

processed_data = run_data_processing_pipeline()

# DATA AUDIT
This layer must evolve to Data Quality Module/ Data audit layer

In [ ]:
# ============================================================
# QUICK VALIDATION
# ============================================================

symbol = list(processed_data.keys())[0]

df = processed_data[symbol]

print(df.columns)
print(df.tail())
print(df.index.min(), "→", df.index.max())

In [ ]:
# ============================================================
# RETURN SANITY CHECK
# ============================================================

import numpy as np

df["vendor_log_ret"] = np.log(df["vendor_adj_close"]).diff()

comparison = df[["log_ret", "vendor_log_ret"]].dropna()

print(comparison.corr())

In [ ]:
# ============================================================
# DATA AUDIT & VALIDATION
# ============================================================

print("\nRunning processed data audit...\n")

audit_rows = []

for file in PROCESSED_DATA_PATH.glob("*.parquet"):

    symbol = file.stem

    df = pd.read_parquet(file)

    start_date = df.index.min()
    end_date = df.index.max()

    rows = len(df)

    duplicates = df.index.duplicated().sum()

    missing_prices = df["Close"].isna().sum()
    missing_adj = df["adj_close"].isna().sum()

    max_return = df["log_ret"].abs().max()

    avg_dollar_volume = df["dollar_volume"].mean()

    audit_rows.append({
        "asset": symbol,
        "start_date": start_date,
        "end_date": end_date,
        "rows": rows,
        "duplicate_dates": duplicates,
        "missing_close": missing_prices,
        "missing_adj_close": missing_adj,
        "max_abs_return": max_return,
        "avg_dollar_volume": avg_dollar_volume
    })

audit_df = pd.DataFrame(audit_rows)

audit_df = audit_df.sort_values("asset")

audit_df

In [ ]:
required_columns = [
"Open","High","Low","Close","Volume",
"vendor_adj_close",
"Dividends","Capital Gains","Stock Splits",
"distribution","dist_factor","cum_adj_factor","adj_close",
"log_ret","log_ret_5","log_ret_21","log_ret_63","log_ret_126","log_ret_252",
"dollar_volume","dollar_volume_21","dollar_volume_63"
]

for file in PROCESSED_DATA_PATH.glob("*.parquet"):

    df = pd.read_parquet(file)

    missing = set(required_columns) - set(df.columns)

    if missing:
        print(file.stem, "missing:", missing)

In [ ]:
max_returns = {}

for file in PROCESSED_DATA_PATH.glob("*.parquet"):

    symbol = file.stem
    df = pd.read_parquet(file)

    max_returns[symbol] = df["log_ret"].abs().max()

pd.Series(max_returns).sort_values(ascending=False)